# Immobilier et performance énergétique en France

Projet du cours Python pour la data science, ENSAE Paris.
Marc Olivier Bouiti Banza, Karlane Vasthi Foudjin Sipeuwo, Aniela Nomwendé Tania Zoungrana.

Ce notebook est le rapport du projet. Il s'exécute du début à la fin à partir des fichiers produits par la chaîne de traitement (voir le README).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from src import chargement, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

ORDRE = chargement.ETIQUETTES
COULEURS = {"A": "#009C6D", "B": "#52B153", "C": "#78BD76", "D": "#F4E70F",
            "E": "#F0B40F", "F": "#EB8235", "G": "#D7221F"}

## Introduction

*À rédiger.* La question, le contexte (réforme du DPE de juillet 2021, loi Climat et résilience, gel des loyers des logements F et G, interdiction de louer les logements G depuis 2025) et l'étude de référence (Chareyron, 2026).

## Partie 1. Les données

### 1.1 Sources et traitement

*À rédiger.* Les trois sources, la chaîne de traitement et les six règles d'appariement (détail en annexe A).

![Chaîne de traitement](../docs/chaine_de_traitement.png)

In [ ]:
ventes = chargement.charger_ventes()
echantillon = chargement.echantillon_analyse(ventes)
print(f"{len(ventes):,} ventes ; {ventes['apparie'].sum():,} appariées ; "
      f"{len(echantillon):,} dans l'échantillon d'analyse")

### 1.2 Le marché

Prix médian au m² de toutes les ventes, appariées ou non, par trimestre et par département.

In [ ]:
prix_trimestre = (ventes.groupby(["trimestre", "type_bien"], observed=True)["prix_m2"]
                  .median()
                  .unstack())

ax = prix_trimestre.plot(marker="o", color={"appartement": "#14213D", "maison": "#A23B3B"})
ax.set_xticks(range(len(prix_trimestre)), prix_trimestre.index, rotation=45, ha="right")
ax.set(title="Prix médian au m² par trimestre (toutes les ventes)", xlabel="", ylabel="€/m²")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
from matplotlib.colors import ListedColormap

# Contours des départements (projet france-geojson de Grégoire David)
URL_DEPARTEMENTS = ("https://raw.githubusercontent.com/gregoiredavid/france-geojson/"
                    "master/departements-version-simplifiee.geojson")
departements = gpd.read_file(URL_DEPARTEMENTS)

prix_dep = ventes.groupby("code_departement", observed=True)["prix_m2"].median().reset_index()
prix_dep["code_departement"] = prix_dep["code_departement"].astype(str)
carte = departements.merge(prix_dep, left_on="code", right_on="code_departement", how="left")

# Classes de prix : sinon Paris (plus de 10 000 €/m²) écraserait toutes les autres couleurs
CLASSES = [0, 1500, 2000, 2500, 3000, 4000, 20000]
NOMS = ["moins de 1 500", "1 500 à 2 000", "2 000 à 2 500", "2 500 à 3 000", "3 000 à 4 000", "plus de 4 000"]
carte["classe_prix"] = pd.cut(carte["prix_m2"], CLASSES, labels=NOMS)

bleus = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS))))
fig, ax = plt.subplots(figsize=(7, 7))
carte.plot(column="classe_prix", cmap=bleus, categorical=True, legend=True, ax=ax,
           edgecolor="white", linewidth=0.3,
           missing_kwds={"color": "lightgrey", "label": "hors champ"},
           legend_kwds={"title": "€/m²", "loc": "center left", "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Prix médian au m² par département, 2022 à 2025")
ax.set_axis_off()
plt.show()

En gris : la Moselle, le Bas-Rhin et le Haut-Rhin, absents de la base des ventes (régime du livre foncier).

*À rédiger.* Commentaire des deux graphiques.

### 1.3 L'échantillon

Part des ventes auxquelles un DPE a été rattaché, puis comparaison des ventes appariées et non appariées. Le détail par département est en annexe B.

In [ ]:
taux = (ventes.groupby(["annee", "type_bien"], observed=True)["apparie"]
        .mean()
        .mul(100)
        .unstack())
taux["ensemble"] = ventes.groupby("annee")["apparie"].mean() * 100
taux.round(1)

In [ ]:
statut = ventes["apparie"].map({True: "appariées", False: "non appariées"})

comparaison = (ventes.groupby(["annee", statut], observed=True)
               .agg(ventes=("prix_m2", "size"),
                    prix_m2_median=("prix_m2", "median"),
                    surface_mediane=("surface", "median"),
                    part_maisons=("type_bien", lambda x: (x == "maison").mean() * 100))
               .round(1))
comparaison.index.names = ["année", "statut"]
comparaison

*À rédiger.* Commentaire : évolution du taux d'appariement et différences entre ventes appariées et non appariées.

## Partie 2. Les résultats

Toutes les analyses de cette partie portent sur l'échantillon d'analyse : les ventes appariées, sans les ventes rapprochées.

### 2.1 Description par étiquette

*À faire.* Tableau par étiquette (prix, surface, type de bien, période de construction), carte de la part de logements F ou G par département, matrice de corrélation de Spearman.

### 2.2 Comparaison sans correction

*À faire.* Boîtes à moustaches du prix au m² par étiquette, maisons et appartements séparés.

### 2.3 Résultat principal : à logement comparable

*À faire.* Écart de prix par étiquette dans des groupes de ventes semblables (commune, type, tranche de surface, période de construction, trimestre), par rapport à G et à D, avec intervalles de confiance. Diagramme en barres : écart sans correction et écart à logement comparable.

### 2.4 Évolution de 2022 à 2025

*À faire.* Le même calcul année par année ; courbe de l'écart des classes F et G avec les dates du gel des loyers et de l'interdiction de louer.

## Partie 3. Vérification et limites

### 3.1 Appariement strict

*À faire.* Le résultat de 2.3 refait sans les ventes ambiguës et sans les ventes appariées par la distance (`chargement.appariement_strict`).

### 3.2 Limites

*À rédiger.* Reprendre les limites de la note de projet.

## Conclusion

*À rédiger.*

## Annexes

### Annexe A. Règles d'appariement

*À rédiger.*

### Annexe B. Taux d'appariement

*À faire.* Tableau par année et type de bien, carte par département.

### Annexe C. Ventes appariées et non appariées

*À faire.* Comparaison détaillée, et répartition des étiquettes comparée à celle du parc de logements.

### Annexe D. Dictionnaire des variables

*À rédiger.*

## Références

*À compléter à partir de la note de projet.*